# Own lagged discharge — does the basin's own flow absorb the neighbour gain?

Camera-ready reviewer request (Area Chair + Reviewer BaCR): *"a comparison with the target basin's
own lagged discharge would help assess the benefit of the neighboring inputs"*.

Pre-registration: `experiments/topology_ablation/preregistrations/own_lag.md` — **read it first**;
the verdict cell checks against the hypotheses written there, before these runs existed.

Three new conditions, 3 seeds each (9 runs, ~45 min each on a T4 → budget ~7 h; the notebook is
idempotent, so a disconnect costs only the run in flight):

| condition | appended input |
|---|---|
| `L_selfQ` | own observed discharge, lagged 1 day |
| `L_selfQ_upQ` | own lagged discharge **+** network neighbour discharge |
| `L_selfQ_knn2` | own lagged discharge **+** nearest-2 neighbour discharge |

Baselines already on Drive: `L`, `L_upQ`, `L_upQknn2`.

**Runtime → Change runtime type → T4 GPU → Run all.** Then paste back Cell 9 and Cell 10.

## Cell 1 — Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Config

In [ ]:
import os
GITHUB_URL='https://github.com/Op-2005/neural_hydro.git'
DRIVE_CAMELS_PATH=''  # blank -> auto-detect
SEEDS=[11,13,17]
AUTO=['/content/drive/MyDrive/datasets/camels_us','/content/drive/MyDrive/neural_hydro/datasets/camels_us',
      '/content/drive/MyDrive/neural_hydrology/datasets/camels_us','/content/drive/MyDrive/camels_us']
if not DRIVE_CAMELS_PATH:
    for c in AUTO:
        if os.path.isdir(c): DRIVE_CAMELS_PATH=c; break
if not DRIVE_CAMELS_PATH or not os.path.isdir(DRIVE_CAMELS_PATH):
    raise RuntimeError(f'CAMELS not found. Tried: {AUTO}')
DRIVE_RUNS='/content/drive/MyDrive/neural_hydro_runs'
os.makedirs(DRIVE_RUNS, exist_ok=True)
print('CAMELS:', DRIVE_CAMELS_PATH); print('RUNS  :', DRIVE_RUNS)

## Cell 3 — Clone repo

In [ ]:
REPO_DIR='/content/nh'; import shutil
%cd /content
if os.path.isdir(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone {GITHUB_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -n 2

## Cell 4 — Install deps (pin numpy<2 / pandas 2.1.4)

In [ ]:
%cd {REPO_DIR}
!pip install -q -e . pynhd networkx 2>&1 | tail -2
!pip install -q --force-reinstall --no-deps "numpy<2" "pandas==2.1.4" 2>&1 | tail -2
import sys
for m in list(sys.modules):
    if m.startswith('numpy') or m.startswith('pandas'): del sys.modules[m]
import numpy as np, pandas as pd, torch
print(f'numpy {np.__version__} pandas {pd.__version__} torch {torch.__version__} CUDA {torch.cuda.is_available()}')

## Cell 5 — Symlink data + runs

In [ ]:
%cd {REPO_DIR}
import shutil
RD=os.path.join(REPO_DIR,'datasets','camels_us'); os.makedirs(os.path.dirname(RD),exist_ok=True)
if os.path.islink(RD): os.unlink(RD)
elif os.path.isdir(RD): shutil.rmtree(RD,ignore_errors=True)
os.symlink(DRIVE_CAMELS_PATH, RD)
RR=os.path.join(REPO_DIR,'runs')
if os.path.islink(RR): os.unlink(RR)
elif os.path.isdir(RR): shutil.rmtree(RR,ignore_errors=True)
os.symlink(DRIVE_RUNS, RR)
os.makedirs(f'{DRIVE_RUNS}/topology_ablation/component0', exist_ok=True)
print('datasets ->', os.path.realpath(RD)); print('runs     ->', os.path.realpath(RR))

## Cell 6 — GPU check

In [ ]:
import torch
if not torch.cuda.is_available(): raise RuntimeError('No GPU. Runtime -> T4 GPU.')
print('GPU:', torch.cuda.get_device_name(0))

## Cell 7 — Build the inputs

`self_q` is the basin's own observed discharge lagged one day, forward-filled over missing days
(CAMELS is near-complete here: 0.05% of training days, 0% of test days). The `upstream_q` files are
rebuilt from the committed edge lists by the same script — verified locally to reproduce the
published network and kNN-2 feature files **bit-for-bit** (max |diff| = 0.000e+00), so these runs
are directly comparable to the existing ones.

In [ ]:
%cd {REPO_DIR}
import pickle
FEAT='experiments/topology_ablation/features'
P1='topology_analysis/phase1_network_discovery/outputs'

def named_ok(p, n_expected=183):
    if not os.path.isfile(p): return False
    d=pickle.load(open(p,'rb'))
    if d[next(iter(d))].index.name!='date': return False
    if len(d)<n_expected:
        print(f'  [rebuild] {os.path.basename(p)} covers {len(d)}/{n_expected} basins'); return False
    return True

SELF=f'{FEAT}/self_q_component0_lag1.p'
NET =f'{FEAT}/upstream_q_network_edgefile_component0_lag1.p'
KNN =f'{FEAT}/upstream_q_knn2_edgefile_component0_lag1.p'

if not named_ok(SELF):
    !python experiments/topology_ablation/build_own_lag_inputs.py --self-lag
if not named_ok(NET):
    !python experiments/topology_ablation/build_own_lag_inputs.py --edge-file {P1}/component0_edges.csv --tag network
if not named_ok(KNN):
    !python experiments/topology_ablation/build_own_lag_inputs.py --edge-file {P1}/component0_edges_knn2.csv --tag knn2
for p in (SELF,NET,KNN):
    print(f'{os.path.basename(p):58s} ok={named_ok(p)}')

## Cell 8 — Train the three conditions × 3 seeds

In [ ]:
%cd {REPO_DIR}
B=f'{REPO_DIR}/runs/topology_ablation/component0'
def done(cond,s): return os.path.isfile(f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv')

JOBS=[('L_selfQ',       [('self_q',SELF)]),
      ('L_selfQ_upQ',   [('self_q',SELF),('upstream_q',NET)]),
      ('L_selfQ_knn2',  [('self_q',SELF),('upstream_q',KNN)])]

for cond,inputs in JOBS:
    for s in SEEDS:
        if done(cond,s): print(f'{cond} seed {s}: done'); continue
        flags=' '.join(f'--extra-input {n} --feature-file {f}' for n,f in inputs)
        print(f'=== training {cond} seed {s} ===')
        !python experiments/topology_ablation/run_own_lag.py --seed {s} --device cuda:0 --epochs 30 --cond-name {cond} {flags}

## Cell 9 — Verdict against the pre-registration

Stdlib only (no pandas/numpy/scipy): the `numpy<2` pin does not reliably take on Colab's Python,
and a verdict cell that crashes after 7 h of training is the worst possible failure. One-sided
Wilcoxon signed-rank by normal approximation with tie correction, which matches
`scipy.stats.wilcoxon` to within 0.5% at n≥150.

In [ ]:
import csv, math, os
B=f'{REPO_DIR}/runs/topology_ablation/component0'

def nse(cond,s):
    p=f'{B}/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
    if not os.path.isfile(p): return None
    out={}
    for r in csv.DictReader(open(p)):
        try: out[str(r['basin']).zfill(8)]=float(r['NSE'])
        except (ValueError,TypeError,KeyError): pass
    return out

def median(v):
    v=sorted(v); n=len(v)
    return v[n//2] if n%2 else 0.5*(v[n//2-1]+v[n//2])

def wilcoxon_greater(d):
    # H1: median(d) > 0
    nz=[x for x in d if x!=0.0]
    n=len(nz)
    if n<10: return float('nan')
    order=sorted(range(n), key=lambda i: abs(nz[i]))
    ranks=[0.0]*n; i=0; ties=[]
    while i<n:
        j=i
        while j+1<n and abs(nz[order[j+1]])==abs(nz[order[i]]): j+=1
        r=0.5*((i+1)+(j+1)); t=j-i+1
        for k in range(i,j+1): ranks[order[k]]=r
        if t>1: ties.append(t)
        i=j+1
    W=sum(ranks[i] for i in range(n) if nz[i]>0)
    mu=n*(n+1)/4.0
    var=n*(n+1)*(2*n+1)/24.0 - sum(t**3-t for t in ties)/48.0
    if var<=0: return float('nan')
    z=(W-mu-0.5)/math.sqrt(var)
    return 0.5*math.erfc(z/math.sqrt(2))

# connected basins = those the network input is nonzero for, read from the edge list
CONN=set()
for r in csv.DictReader(open(f'{REPO_DIR}/topology_analysis/phase1_network_discovery/outputs/component0_edges.csv')):
    CONN.add(str(r['child_id']).zfill(8))
CONN=sorted(CONN)
ALL=sorted(nse('L',SEEDS[0]).keys())
print(f'connected basins: {len(CONN)}   all basins: {len(ALL)}')

def contrast(a,b,basins,label):
    per=[]; ps=[]
    for s in SEEDS:
        A,Bv=nse(a,s), nse(b,s)
        if A is None or Bv is None: print(f'  [missing] {a} or {b} seed {s}'); return None
        d=[A[x]-Bv[x] for x in basins if x in A and x in Bv]
        per.append(median(d)); ps.append(wilcoxon_greater(d))
    cross=sum(per)/len(per); weak=max(ps)
    print(f'{label:46s} {cross:+.4f}   per-seed {" / ".join(f"{x:+.4f}" for x in per)}   weakest p={weak:.3g}')
    return cross, per, weak

print('\n--- H1: does own lagged flow help on its own? (connected n=%d) ---'%len(CONN))
h1=contrast('L_selfQ','L',CONN,'L_selfQ - L')
contrast('L_selfQ','L',ALL,'L_selfQ - L  (all 183)')

print('\n--- reference: the published neighbour gains ---')
contrast('L_upQ','L',CONN,'L_upQ - L (network)')
contrast('L_upQknn2','L',CONN,'L_upQknn2 - L (nearest 2)')

print('\n--- H2 (DECISIVE): does own flow absorb the neighbour gain? ---')
h2net=contrast('L_selfQ_upQ','L_selfQ',CONN,'network increment over self_q')
h2knn=contrast('L_selfQ_knn2','L_selfQ',CONN,'nearest-2 increment over self_q')

print('\n=== PRE-REGISTERED VERDICT (preregistrations/own_lag.md) ===')
for name,res in (('network',h2net),('nearest-2',h2knn)):
    if res is None: continue
    cross,per,weak=res
    survives = all(x>0 for x in per) and weak<0.05
    if survives:
        print(f'  {name}: INCREMENT SURVIVES ({cross:+.4f}, all seeds positive, weakest p={weak:.3g}).')
        print('     -> neighbour discharge carries spatial information BEYOND the target\'s own state.')
    elif cross < 0.005:
        print(f'  {name}: INCREMENT COLLAPSES ({cross:+.4f}, weakest p={weak:.3g}).')
        print('     -> most of the neighbour signal is recoverable from the basin\'s own history.')
        print('        C1/C5/C6 unaffected (all are contrasts among neighbour selectors); report as scope.')
    else:
        print(f'  {name}: AMBIGUOUS ({cross:+.4f}, weakest p={weak:.3g}) - positive but not at every seed.')
if h2net and h2knn:
    print(f'\n  H3 ordering (expect nearest-2 >= network): {h2knn[0]:+.4f} vs {h2net[0]:+.4f}'
          f' -> {"consistent" if h2knn[0]>=h2net[0] else "INCONSISTENT with proximity"}')

## Cell 10 — Persistence check (did the runs reach Drive?)

In [ ]:
print('=== persistence (in Drive?) ===')
missing=[]
for cond,_ in JOBS:
    for s in SEEDS:
        dp=f'{DRIVE_RUNS}/topology_ablation/component0/{cond}_component0_seed{s}/test/model_epoch030/test_metrics.csv'
        ok=os.path.isfile(dp); print(f'  {cond} seed {s}: {ok}')
        if not ok: missing.append((cond,s))
print('\nALL PERSISTED' if not missing else f'\nMISSING: {missing}  (re-run Cell 8; if the symlink failed, use files.download())')

## Done

Paste back **Cell 9** (the verdict) and **Cell 10** (persistence).

Either outcome is reportable and neither threatens the paper's thesis — C1, C5 and C6 are contrasts
among neighbour selectors and do not depend on the increment over an autoregressive baseline. The
pre-registration states both readings in advance.